# Multimodal ETL: Vision Processing, Structured Data Extraction, and Validation Routing

**Objective:** Design a deterministic processing pipeline that ingests raw, messy image scans (e.g., KYC Identity Documents), utilizes multimodal Vision models for Optical Character Recognition (OCR), and extracts strictly validated JSON records for downstream banking systems. This audit evaluates API cost implications, JSON schema enforcement vs. Pydantic validation, and Human-in-the-Loop exception handling.

In [ ]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from IPython.display import Image, display
try:
    from providers_utils import groq_client, ask            # Session 11
    from local_models_utils import PROBE_TOOLS              # Session 12 (used in section 7)
    from vision_utils import (SAMPLE_DIR, KYC_JSON_SCHEMA, KycRecord, TRANSCRIBE_PROMPT, compare_to_truth, encode_image,
                              extract_validated, find_vision_model, parse_and_validate, read_image_text,
                              sample_transcript, success_rate)
except ImportError as e:
    raise ImportError(f"{e}. This notebook uses files from Sessions 11, 12 and 13: run 'git pull' in the course folder, "
                      "then restart the kernel.") from e

client = groq_client()
TEXT_MODEL = os.getenv("MODEL")
VISION = os.getenv("VISION_MODEL") or find_vision_model(client)
print("text model  :", TEXT_MODEL)
print("vision model:", VISION or "none found - sections 2-3 will use the saved transcripts instead")

## 1. Visual Tokenization & API Cost Implications
A vision model converts an image file into a base64 string, adding it to the message payload. We benchmark the token disparity between a pure text prompt and a multimodal image payload.

In [ ]:
display(Image(filename=str(SAMPLE_DIR / "doc1_id_card.png"), width=480))
url = encode_image(SAMPLE_DIR / "doc1_id_card.png")
print("the image as a string:", url[:60], "...", f"({len(url):,} characters)")

INSTRUCTION = TRANSCRIBE_PROMPT      
if VISION:
    with_image = read_image_text(client, VISION, SAMPLE_DIR / "doc1_id_card.png")
    text_only = ask(client, VISION, "", INSTRUCTION, max_tokens=20)
    print("prompt tokens, text only      :", text_only["prompt_tokens"])
    print("prompt tokens, text + image   :", with_image["prompt_tokens"])
    print("so the image added about       :", with_image["prompt_tokens"] - text_only["prompt_tokens"], "tokens")
else:
    print("skipped (no vision model)")

**Key Finding — Exponential Cost of Vision:** Processing scanned documents as images causes daily operational costs to increase exponentially, as the model breaks each image into a grid of visual tiles that consume significantly more tokens than standard text. To mitigate this in a high-volume environment (e.g., 20,000 documents/day), the workload must be routed to an asynchronous batch-processing queue utilizing high-latency, low-cost API tiers rather than premium, real-time endpoints.

## 2. OCR Fallibility & Alphanumeric Vulnerability
We evaluate the OCR capabilities of the Vision model on a heavily degraded, rotated, and noisy ID scan against a ground-truth transcript.

In [ ]:
def read(stem):
    truth = sample_transcript(stem)
    if not VISION:
        return truth, "ground-truth transcript (no vision model available)"
    try:
        return read_image_text(client, VISION, SAMPLE_DIR / f"{stem}.png")["text"], f"read by {VISION}"
    except Exception as e:
        return truth, f"ground-truth transcript (vision call failed: {type(e).__name__})"

for stem in ("doc1_id_card", "doc3_messy_scan"):
    text, source = read(stem)
    print(f"=== {stem}  [{source}]\n{text.strip()}\n")
    if source.startswith("read by"):                       # only meaningful when a model really read the image
        print("    check against the saved ground truth:")
        for line, ok in compare_to_truth(text, sample_transcript(stem)):
            print(f"      {'match    ' if ok else 'DIFFERENT'}  {line}")
        print()

**Key Finding — The Alphanumeric Risk:** While the logic engine extracted the correct ID number despite heavily fractured text (e.g., "TY CARD"), messy scans expose extreme vulnerabilities in characters like O/0 and Z/2. Because random alphanumeric ID strings lack the semantic context of natural language, a single misread character cannot be auto-corrected, causing automated KYC verification pipelines to fail immediately.

## 3. Strict Schema Enforcement vs. Factual Accuracy
We demand structured JSON output using three increasingly strict API mechanisms: a raw text prompt, `json_object` mode, and strict `json_schema` enforcement.

In [ ]:
RUNS = 5
doc_text = sample_transcript("doc1_id_card")
print(f"{'level':12} {'passed':>8}")
for mode in ("prompt", "json_mode", "schema"):
    r = success_rate(client, TEXT_MODEL, doc_text, mode, runs=RUNS)
    print(f"{mode:12} {r['ok']:>4}/{r['runs']}")

**Key Finding — Syntactic vs. Semantic Guarantees:** While enforcing a strict JSON schema mathematically guarantees the syntactic shape, keys, and data types of the output, it completely fails to guarantee factual accuracy; the model will adhere perfectly to the structural requirements while actively hallucinating false data values.

## 4. Pydantic Business Logic vs. LLM Capabilities
A schema ensures a field like `date_of_birth` is a string. It takes deterministic, server-side Python code (`Pydantic`) to evaluate if that string represents a valid, non-future date that satisfies banking regulations.

In [ ]:
BASE = {"document_type": "id_card", "full_name": "TEST PERSON", "date_of_birth": "1994-03-14",
        "id_number": "ABCDE1234F", "address": None}

cases = {
    "date of birth in the future": {**BASE, "date_of_birth": "2999-01-01"},     # a string, so the schema is happy
    "ID number in the wrong pattern": {**BASE, "id_number": "12345ABCDE"},      # a string, so the schema is happy
    "ID card with no date of birth": {**BASE, "date_of_birth": None},           # null is allowed by the schema
}
for label, record in cases.items():
    try:
        KycRecord(**record)
        print(f"{label}: accepted (unexpected)")
    except Exception as e:
        print(f"{label}:\n    rejected -> {str(e).splitlines()[-2].strip() if len(str(e).splitlines())>1 else e}")


**Key Finding — The Limits of Autonomous AI:** Enterprise workflows require a layered architecture. The LLM schema guarantees the exact structural extraction (e.g., pulling the `expiration_date`). Deterministic code (Pydantic) handles the mathematical application logic (e.g., verifying the date is greater than today). Finally, Human-in-the-Loop checks are mandatory for physical reality validation, such as confirming the live applicant matches the photo.

## 5. Exception Handling & The Human-in-the-Loop
When validation fails, we inject the specific error back into the model for a single retry. If the retry fails because the text was inherently misread by the OCR step, the pipeline must fail safely.

In [ ]:
clean = sample_transcript("doc3_messy_scan")
misread = clean.replace("PQRST5678Z", "PQRST56782")     # last character read as the digit 2

for label, text in (("clean transcript", clean), ("misread transcript", misread)):
    out = extract_validated(client, TEXT_MODEL, text, mode="schema", max_retries=1)
    status = "OK" if out["record"] else "FAILED -> route to a human reviewer"
    print(f"--- {label}: {status} after {out['attempts']} attempt(s)")
    for e in out["errors"]:
        print("    error fed back:", e[:140])
    if out["record"]:
        print("    ", out["record"].model_dump())

**Key Finding — Safe Failure Routing:** The correct production behavior is to stop the automated pipeline after one retry and route the record to a manual exception queue. Endless retries waste API tokens and actively prompt the model to hallucinate fake data just to satisfy the validator. An autonomous agent must never have final execution authority over high-risk, unrecoverable tasks when validation fails.

## 6. Conditional Business Logic (Utility Bills vs. ID Cards)
Different document types have different required fields. A utility bill does not contain a Date of Birth or an ID number, meaning those fields will return `null`. We test the pipeline's ability to handle conditionally required data.

In [ ]:
text, source = read("doc2_utility_bill")
print(f"[{source}]\n{text.strip()}\n")
out = extract_validated(client, TEXT_MODEL, text, mode="schema", max_retries=1)
print("attempts:", out["attempts"])
print(out["record"].model_dump() if out["record"] else out["errors"])

**Key Finding — Conditional Validation Routing:** The JSON Schema simply defines the data type, allowing `date_of_birth` to be passed as `null` across the board. It is the Pydantic model that enforces the conditional business logic: it dynamically accepts `null` if the `document_type` is a utility bill, but strictly rejects it if the document is an ID card. This proves that structural extraction (LLM/JSON Schema) and conditional business rules (Python/Pydantic) must be kept strictly separated to build scalable, maintainable architectures.